In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import graphinglib as gl
import pandas as pd
import pyregion
import palettable
import pvextractor
from PIL import Image
from astropy.constants import c as speed_of_light
from skimage.feature import peak_local_max

from src.hdu.map import Map
from src.hdu.cube import Cube
from src.coordinates.celestial_coords import RA, DEC
from src.tools.plotting import *
from src.config import REDSHIFT

In [ ]:
pc_per_px = 20.6/2  # pc/px at the high resolution
arcsec_per_px = 0.05  # arcsec/px at the high resolution

## PV diagram

In [ ]:
from astropy.io import fits

f = fits.open("data/letter/q3dfit_ngc4696_v7_choiclip1_1a_marquis1_paa_g1_3ba_paa_v50_1_newwcs.fits")
velocity_field = Map.from_hdu(f[1]).mask(f[2].data)
hm = velocity_field.data.plot.copy_with(color_map="coolwarm", show_color_bar=False, color_map_range=(-600, 600))

AGN_pos = get_AGN_pos(velocity_field.header)

In [ ]:
unsubtracted_data = Cube.load("data/letter/ngc4696_v7_choiclip1_1a_marquis1-conv_2.5_paa_contsub_1.fits", hdu_index=1).data[575:610]
cube = Cube.load("data/letter/ngc4696_v7_choiclip1_1a_marquis1-conv_2.5_paa_contsub_1.fits", hdu_index=5)
data, wavelengths = cube.data, cube.header.wavelengths / (1 + REDSHIFT)

noise_map = np.nanstd(data[615:650, :, :], axis=0)
noise = np.nanmean(
    Map(noise_map, header=cube.header.celestial).get_masked_region(pyregion.open("data/letter/confidence_region.reg")).data
)
cmap_levels = np.arange(3 * noise, 17 * noise, noise)

aperture_colors = palettable.cartocolors.qualitative.Vivid_3.mpl_colors
paths = pvextractor.paths_from_regfile("data/letter/CNDFilament_v5_shorter.reg")
apertures_df = pd.DataFrame(columns=["points", "width", "color"], data=list(zip(
    [path.get_xy(cube.header.wcs) for path in paths],
    [3, 5],
    aperture_colors[1:],
)))

x_tick_spacing_pc = 100  # pc
y_tick_spacing = 400  # km/s
x_tick_spacing_arcsec = 0.5  # arcsec

cropped_data = data[575:610, :, :]
rest_wavelength = 1.87561e-6

# Calculate velocities for this line
wavelengths_slice = wavelengths[575:610]
velocities = (wavelengths_slice - rest_wavelength) / rest_wavelength * speed_of_light.to("kilometer/s").value

velocity_tick_values = np.arange(
    np.ceil(np.min(velocities) / y_tick_spacing) * y_tick_spacing,
    np.max(velocities),
    y_tick_spacing
)
# Map velocity values to pixel indices
velocity_tick_positions = np.interp(velocity_tick_values, velocities, np.arange(len(velocities)))
zero_velocity_line = gl.Hlines(
    velocity_tick_positions[velocity_tick_values == 0],
    colors="black",
    line_widths=1,
    line_styles="--",
    alpha=0.5,
)

cumulative_length_pc = 0  # Track cumulative length in parsecs
aperture_lengths_px = []
figs = []
apertures = []

for i, aperture in enumerate(apertures_df.itertuples()):
    aperture_poly, bin_polys, aperture_arrow, pv_hm, pv_cont_filled, pv_cont_lines = make_pv_diagram(
        data_cube=cropped_data,
        wcs=cube.header.wcs,
        aperture=aperture.points,
        width=aperture.width,
        spacing=1,
        contour_levels=cmap_levels,
    )


    apertures.extend([
        *[poly.copy_with(edge_color=aperture.color) for poly in [aperture_poly, *bin_polys]],
        aperture_arrow.copy_with(color=aperture.color)
    ])

    # Individual figures
    fig = gl.SmartFigure(elements=[pv_cont_filled, pv_cont_lines, zero_velocity_line], reference_labels_loc="inside")
    fig.set_visual_params(axes_edge_color=aperture.color, axes_line_width=2)
    fig.set_tick_params(which="both", draw_left_labels=False, draw_top_ticks=True, draw_right_ticks=True)
    figs.append(fig)

    # Scaling
    aperture_points = np.array(aperture.points)
    current_aperture_length_px = np.linalg.norm(aperture_points[1] - aperture_points[0])
    current_aperture_length_pc = current_aperture_length_px * pc_per_px

    # Calculate where ticks should appear in parsec space (x-axis)
    tick_positions_pc = np.arange(
        np.ceil(cumulative_length_pc / x_tick_spacing_pc) * x_tick_spacing_pc,
        cumulative_length_pc + current_aperture_length_pc,
        x_tick_spacing_pc
    )
    tick_positions_px = (tick_positions_pc - cumulative_length_pc) / pc_per_px

    fig.set_ticks(
        x_ticks=tick_positions_px,
        x_tick_labels=list(map(round, tick_positions_pc)),
        y_ticks=velocity_tick_positions,
        y_tick_labels=list(map(round, velocity_tick_values))
    )

    cumulative_length_arcsec = cumulative_length_pc / pc_per_px * arcsec_per_px
    tick_positions_arcsec = np.arange(
        np.ceil(cumulative_length_arcsec / x_tick_spacing_arcsec) * x_tick_spacing_arcsec,
        cumulative_length_arcsec + current_aperture_length_pc / pc_per_px * arcsec_per_px,
        x_tick_spacing_arcsec
    )
    tick_positions_arcsec_px = (tick_positions_arcsec - cumulative_length_arcsec) / arcsec_per_px

    # Add blue/red lines for the peaks in the first aperture
    if i == 0:
        coordinates = peak_local_max(pv_hm.image, min_distance=1, num_peaks=2).astype(float)
        coordinates = np.sort(coordinates, axis=0)  # Sort by x-coordinate
        fig.add_elements([gl.Vlines(coordinates[:, 1], colors=["blue", "red"], line_widths=0.8, line_styles="-")])
        unsubtracted_pv_data = make_pv_diagram(
            data_cube=unsubtracted_data,
            wcs=cube.header.wcs,
            aperture=aperture.points,
            width=aperture.width,
            spacing=1,
            contour_levels=cmap_levels,
        )[3].image
        x_data = np.arange(unsubtracted_pv_data.shape[0])
        red_spectrum = gl.Curve(x_data, unsubtracted_pv_data[:, int(coordinates[1, 1])], color="red")
        blue_spectrum = gl.Curve(x_data, unsubtracted_pv_data[:, int(coordinates[0, 1])], color="blue")
        for i, (spectrum, color, aperture_i) in enumerate(
            zip([red_spectrum, blue_spectrum], ["red", "blue"], coordinates[::-1,1])
        ):
            spec_fig = gl.SmartFigure(
                remove_x_ticks=True,
                remove_y_ticks=True,
                elements=[spectrum],
                reference_labels_loc="inside",
            )
            nested_fig = gl.SmartFigure(elements=[spec_fig], size=(1.1, 0.7)) # Use nested SmartFigure to create a reference label
            nested_fig.set_reference_labels_params(start_index=3+i)
            nested_fig.save(f"figures/letter/{color}_spectrum.png", dpi=600, transparent=True)

            # Change the color of the relevant apertures
            aperture = apertures[int(aperture_i)]
            aperture.fill = True
            aperture.fill_color = color
            aperture.edge_color = color
            aperture.fill_alpha = 1

    fig.create_twin_axis(
        is_y=False,
        elements=[pv_cont_filled.copy_with(alpha=0)]
    ).set_ticks(
        ticks=tick_positions_arcsec_px,
        tick_labels=list(map(lambda x: f"{x:.1f}", tick_positions_arcsec))
    )

    cumulative_length_pc += current_aperture_length_pc
    aperture_lengths_px.append(current_aperture_length_pc / pc_per_px)

figs[0].set_tick_params(draw_left_labels=True, draw_left_ticks=True)
figs[0][0] += [gl.Text(1, 2, r"\textbf{East}", h_align="left")]
resolution_line = gl.Curve([5, 7], [5, 5], color="black", line_width=1)  # 0.1" line
resolution_line.add_errorbars(y_error=[1, 1], cap_width=0, cap_thickness=10)
figs[1][0] += [gl.Text(35, 2, r"\textbf{West}", h_align="right"),
               resolution_line,
               gl.Text(6, 7, "0.1\"", h_align="center")]
figs[0].set_reference_labels_params(start_index=1)
figs[1].set_reference_labels_params(start_index=2)
# figs[1][0] += [gl.PlottableAxMethod("annotate", text="", xy=(5, 4.5), xycoords='data', xytext=(7, 4.5), arrowprops={"arrowstyle": "-"}, ha="center", va="bottom")]
# plt.annotate(text="", xy=(0, 0.5), xytext=(1, 0.5), arrowprops={"arrowstyle": "-"})

pv_fig = gl.SmartFigure(
    num_cols=len(figs),
    x_label="Position along the aperture [pc]",
    y_label=r"Pa$\alpha$ velocity [km s$^{{-1}}$]",
    title="Position along the aperture [arcsec]",
    elements=figs,
    width_padding=0.0018,
    width_ratios=aperture_lengths_px
)

fig = gl.SmartFigure(
    num_cols=2,
    remove_x_ticks=True,
    remove_y_ticks=True,
    aspect_ratio=[1],
    size=(10, 3),
    x_lim=(30, None),
    y_lim=(25, None),
    width_ratios=[1, 2],
    reference_labels=True,
    reference_labels_loc="inside",
    elements=[[hm, *apertures, AGN_pos, *get_N_E_arrows(theta=0, center=(105, 30), arrow_length=10, arrow_offset=0.5)],
              pv_fig],
)

fig.save("figures/letter/pv_diagram.png", dpi=600)

# Add spectra to the PV diagram
pv_fig = Image.open("figures/letter/pv_diagram.png").convert("RGBA")
blue_spectrum = Image.open("figures/letter/blue_spectrum.png").convert("RGBA")
red_spectrum = Image.open("figures/letter/red_spectrum.png").convert("RGBA")

pv_fig.alpha_composite(blue_spectrum, dest=(4100, 1050))
pv_fig.alpha_composite(red_spectrum, dest=(4100, 300))

pv_fig.save("figures/letter/pv_diagram_with_spectra.png", dpi=(600, 600))

In [ ]:
gl.SmartFigure(elements=[
    gl.Scatter([0, 1], [0, 1], label="Test", face_color="orange"),
    gl.Line((0, 0.5), (1, 0.5), capped_line=True)
]).show()